In [1]:
import requests
try:
    r = requests.get("https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/ethanol/property/CanonicalSMILES/TXT",
                     timeout=15)
    print("STATUS", r.status_code, "| BODY", r.text.strip()[:80])
except Exception as e:
    print("NO INTERNET FROM THIS NODE:", type(e).__name__, str(e)[:120])

STATUS 200 | BODY CCO


In [1]:
import pandas as pd, os

DATA_DIR = "."   # <- set to the folder holding the three CSVs on CCAST
FILES = {
    "1947": "1947-King-USDA_Dataset.csv",
    "1954": "1954-King_dataset.csv",
    "1967": "1967_USDA_datasetcsv.csv",
}

def read_any(path):
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(path, encoding=enc, dtype=str), enc
        except Exception as e:
            last = e
    raise last

for year, fname in FILES.items():
    path = os.path.join(DATA_DIR, fname)
    print("=" * 70)
    print(f"{year}: {fname}")
    if not os.path.exists(path):
        print("  !! NOT FOUND at this path"); continue
    df, enc = read_any(path)
    print(f"  rows={len(df)}  cols={len(df.columns)}  encoding={enc}")
    print(f"  columns: {list(df.columns)}")
    print("  non-null counts:")
    print(df.notna().sum().to_string().replace('\n', '\n    '))
    print("  first 3 rows:")
    print(df.head(3).to_string()[:1500])

1947: 1947-King-USDA_Dataset.csv
  rows=7089  cols=4  encoding=utf-8
  columns: ['Chemical', 'Repellent_skin_YF', 'Repellent_cloth_YF', 'Unnamed: 3']
  non-null counts:
Chemical              7089
    Repellent_skin_YF     4106
    Repellent_cloth_YF    3133
    Unnamed: 3               1
  first 3 rows:
                                             Chemical Repellent_skin_YF Repellent_cloth_YF Unnamed: 3
0    Abietic acid, diethylene glycol ester (Flexalyn)                 1                NaN        NaN
1                           Abietic acid, ethyl ester                 1                NaN        NaN
2  Abietic acid, hydrogenated methyl ester (Hercolyn)                 1                NaN        NaN
1954: 1954-King_dataset.csv
  rows=8201  cols=5  encoding=utf-8
  columns: ['Chemical', 'Repellency_skin_YF', 'Repellency_skin_M', 'Repellency_clothes_YF', 'Repellency_clothes_M']
  non-null counts:
Chemical                 8201
    Repellency_skin_YF       8201
    Repellency_skin_M   

# Cell 2: build the tidy long observation table from all three files.
It maps the name column, preserves every trait value verbatim under a year prefix, captures then drops the junk column, and previews how much the unique-name dedup will save on API calls.

In [2]:
import pandas as pd, os, re
from collections import Counter

DATA_DIR = "."  # same folder as Cell 1
SPEC = {
    "1947": ("1947-King-USDA_Dataset.csv", "Chemical"),
    "1954": ("1954-King_dataset.csv",      "Chemical"),
    "1967": ("1967_USDA_datasetcsv.csv",   "MATERIAL"),
}

def read_any(path):
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try: return pd.read_csv(path, encoding=enc, dtype=str)
        except Exception as e: last = e
    raise last

frames = []
for year, (fname, name_col) in SPEC.items():
    df = read_any(os.path.join(DATA_DIR, fname))

    # surface then drop stray 'Unnamed' artifact columns
    junk = [c for c in df.columns if str(c).startswith("Unnamed")]
    for c in junk:
        vals = df[c].dropna().tolist()
        print(f"[{year}] dropping junk column {c!r}; non-null values seen: {vals[:5]}")
    df = df.drop(columns=junk)

    trait_cols = [c for c in df.columns if c != name_col]
    out = pd.DataFrame({
        "source_year": year,
        "source_row_id": [f"{year}_{i}" for i in range(len(df))],
        "original_name": df[name_col].astype(str).str.strip(),
    })
    # keep every trait verbatim, namespaced by year so nothing collides or is reinterpreted
    for c in trait_cols:
        out[f"y{year}_{c}"] = df[c]
    frames.append(out)

obs = pd.concat(frames, ignore_index=True)
print(f"\nObservation table: {len(obs)} rows  |  {obs['source_year'].value_counts().to_dict()}")
print(f"Columns: {list(obs.columns)}")

# --- API-savings preview (NAIVE key only; real normalization comes next cell) ---
naive = obs["original_name"].str.lower().str.strip()
per_name_years = obs.assign(k=naive).groupby("k")["source_year"].nunique()
print(f"\nTotal occurrences:      {len(obs)}")
print(f"Unique (naive key):     {naive.nunique()}")
print(f"Names in >=2 datasets:  {(per_name_years>=2).sum()}")
print(f"Names in all 3:         {(per_name_years==3).sum()}")
print("\nSample rows:")
print(obs.head(4).to_string())

[1947] dropping junk column 'Unnamed: 3'; non-null values seen: ['1']

Observation table: 23881 rows  |  {'1967': 8591, '1954': 8201, '1947': 7089}
Columns: ['source_year', 'source_row_id', 'original_name', 'y1947_Repellent_skin_YF', 'y1947_Repellent_cloth_YF', 'y1954_Repellency_skin_YF', 'y1954_Repellency_skin_M', 'y1954_Repellency_clothes_YF', 'y1954_Repellency_clothes_M', 'y1967_Repellent_cloth_YF', 'y1967_Repellent_skin_YF', 'y1967_Repellent_wash_YF', 'y1967_Repellent_space_YF']

Total occurrences:      23881
Unique (naive key):     23293
Names in >=2 datasets:  536
Names in all 3:         0

Sample rows:
  source_year source_row_id                                       original_name y1947_Repellent_skin_YF y1947_Repellent_cloth_YF y1954_Repellency_skin_YF y1954_Repellency_skin_M y1954_Repellency_clothes_YF y1954_Repellency_clothes_M y1967_Repellent_cloth_YF y1967_Repellent_skin_YF y1967_Repellent_wash_YF y1967_Repellent_space_YF
0        1947        1947_0    Abietic acid, diethyl

# Cell 3 builds the real (but deliberately conservative) normalization key. 
It only neutralizes case, Unicode, whitespace, and comma-spacing — it never truncates, reorders, or drops a substituent, because over-merging before lookup would fuse distinct compounds, and structure resolution is what's supposed to do the heavy equivalencing. It also builds the worklist (key → all the occurrences it covers) so we resolve each unique name once and fan the result back out, and it prints a few groups that merged beyond exact-string so you can eyeball that the key isn't fusing things it shouldn't.

In [3]:
import unicodedata, re

def norm_key(name):
    s = unicodedata.normalize("NFKC", str(name))
    s = s.strip().lower()
    s = (s.replace("\u2013", "-").replace("\u2014", "-")
           .replace("\u2019", "'").replace("\u201c", '"').replace("\u201d", '"'))
    s = re.sub(r"\s*,\s*", ", ", s)   # standardize comma spacing
    s = re.sub(r"\s+", " ", s)        # collapse whitespace
    s = s.strip().strip(".")          # trim trailing period
    return s

obs["norm_key"] = obs["original_name"].map(norm_key)

# worklist: one entry per unique key, with the occurrences it covers
worklist = (obs.groupby("norm_key")
                .agg(n_occurrences=("source_row_id", "size"),
                     years=("source_year", lambda s: sorted(set(s))),
                     example_names=("original_name", lambda s: sorted(set(s))[:3]),
                     row_ids=("source_row_id", list))
                .reset_index())

per_key_years = worklist.set_index("norm_key")["years"].map(len)
print(f"Total occurrences:          {len(obs)}")
print(f"Unique (naive lower/strip): {obs['original_name'].str.lower().str.strip().nunique()}")
print(f"Unique (conservative key):  {obs['norm_key'].nunique()}")
print(f"Keys in >=2 datasets:       {(per_key_years>=2).sum()}")
print(f"Keys in all 3 datasets:     {(per_key_years==3).sum()}")
print(f"API calls saved vs naive:   "
      f"{obs['original_name'].str.lower().str.strip().nunique() - obs['norm_key'].nunique()}")

# show keys whose distinct ORIGINAL spellings collapsed (sanity check: same compound?)
collapsed = (obs.groupby("norm_key")["original_name"].nunique()
                .sort_values(ascending=False))
print("\nKeys that merged >1 distinct original spelling (top 10):")
for k, n in collapsed[collapsed > 1].head(10).items():
    spellings = sorted(obs.loc[obs["norm_key"] == k, "original_name"].unique())
    print(f"  [{n}] {spellings}")

Total occurrences:          23881
Unique (naive lower/strip): 23293
Unique (conservative key):  23270
Keys in >=2 datasets:       558
Keys in all 3 datasets:     0
API calls saved vs naive:   23

Keys that merged >1 distinct original spelling (top 10):
  [2] ['p-Nitrobenzoic acid, ethyl ester', 'p-Nitrobenzoic acid,ethyl ester']
  [2] ['1, 2-Tetradecanediol', '1,2-Tetradecanediol']
  [2] ['Succinic acid,  monoethyl ester', 'Succinic acid, monoethyl ester']
  [2] ['Succinic acid,  monoallyl ester', 'Succinic acid, monoallyl ester']
  [2] ['Cinnamamide', 'cinnamamide']
  [2] ['Cinnamamide, p-chloro-N, N-diethyl-', 'Cinnamamide, p-chloro-N,N-diethyl-']
  [2] ['2-Octadecynoic acid, methyl ester', '2-octadecynoic acid, methyl ester']
  [2] ['Succinic acid,  diallyl ester', 'Succinic acid, diallyl ester']
  [2] ['4-Cyclohexene-1,2-dicarboxylic acid, dimethyl ester', '4-Cyclohexene-1,2-dicarboxylic acid,dimethyl ester']
  [2] ['Succinic acid,  dibenzyl ester', 'Succinic acid, dibenzyl ester']

# Cell 4 — the claims.py module, which doubles as a runnable cell (the __main__ battery prints the table above so you can re-verify on your machine

In [4]:
from __future__ import annotations
import re
from dataclasses import dataclass, field

# ---- element morphemes: presence is high-confidence ----
ELEMENT_MORPHEMES = {
    "S":  r"thio|mercapt|sulf|sulph|thia|thiol",
    "N":  r"amino|amine|ammonium|amide|imide|imine|nitro|nitrile|cyano|azo|hydrazi|oxime|\burea\b|carbamate|pyridine|morpholine|piperidine|piperazine|quinoline",
    "Cl": r"chloro|chloride",
    "Br": r"bromo|bromide",
    "F":  r"fluoro|fluoride",
    "I":  r"\biodo|iodide",
    "P":  r"phospho|phosph",
    "Si": r"\bsila|silane|silox|silyl",
}

ESTER_ATE = (r"acetate|benzoate|propionate|propanoate|butyrate|butanoate|laurate|"
             r"palmitate|stearate|oleate|salicylate|phthalate|citrate|lactate|"
             r"tartrate|succinate|maleate|fumarate|abietate|formate|carbonate")
SALT_WORDS = (r"sodium|potassium|calcium|magnesium|ammonium|zinc|copper|cupric|cuprous|"
              r"ferric|ferrous|iron|lithium|barium|aluminum|aluminium|lead|mercur|silver|"
              r"\bsalt\b|hydrochloride|hydrobromide|\bhcl\b|sulfate|sulphate|nitrate\b")
MIXTURE_WORDS = r"\bmixture\b|\bmixed\b|\bblend\b|\bcondensate\b|reaction product"

@dataclass
class ClaimSet:
    connectivity: str = "single"          # single | salt | mixture
    functional_groups: set = field(default_factory=set)
    elements: set = field(default_factory=set)
    notes: list = field(default_factory=list)

def _f(pat, s):
    return re.search(pat, s, re.I) is not None

def extract_claims(name: str) -> ClaimSet:
    raw = str(name).strip()
    low = raw.lower()
    cs = ClaimSet()

    # ---------- connectivity ----------
    if _f(MIXTURE_WORDS, low):
        cs.connectivity = "mixture"; cs.notes.append("mixture keyword")
    if _f(SALT_WORDS, low):
        cs.connectivity = "salt"; cs.notes.append("salt keyword -> multi-fragment allowed")
    is_salt = cs.connectivity == "salt"

    # ---------- functional groups ----------
    ester_hit = (_f(r"\bacid\b.*\bester\b", low) or _f(r"\bester\s+with\b", low)
                 or _f(r"\b(mono|di|tri|tetra)?ester\b", low) or _f(ESTER_ATE, low))
    if ester_hit and not is_salt:
        cs.functional_groups.add("ester"); cs.notes.append("ester linkage implied")
    if _f(ESTER_ATE, low) and is_salt:
        cs.functional_groups.add("carboxylate"); cs.notes.append("acylate + salt -> carboxylate")

    if _f(r"mercaptal|dithioacetal", low):
        cs.functional_groups.add("dithioacetal"); cs.notes.append("S,S-acetal (mercaptal)")
    elif _f(r"mercaptol|dithioketal", low):
        cs.functional_groups.add("dithioketal"); cs.notes.append("S,S-ketal (mercaptol)")
    elif _f(r"\bacetal\b", low):
        cs.functional_groups.add("acetal"); cs.notes.append("O,O-acetal")
    elif _f(r"\bketal\b", low):
        cs.functional_groups.add("ketal")

    if low.endswith("amide") or _f(r"amide|\bamid\b|lactam", low):
        cs.functional_groups.add("amide")
    if _f(r"\bamine\b|amino", low):
        cs.functional_groups.add("amine")
    if _f(r"\bnitro\b|dinitro|trinitro", low):
        cs.functional_groups.add("nitro")
    if _f(r"alcohol|carbinol|\bdiol\b|\btriol\b|glycol|glycerol|enediol|anediol", low):
        cs.functional_groups.add("alcohol")
    if _f(r"\bketone\b|quinone", low):
        cs.functional_groups.add("ketone")
    if _f(r"phenol|cresol|xylenol|catechol|resorcinol|hydroquinone", low):
        cs.functional_groups.add("phenol")

    if _f(r"\bacid\b", low) and "ester" not in low and not is_salt and "acetate" not in low:
        cs.functional_groups.add("carboxylic_acid")
    if is_salt and _f(r"\bacid\b", low):
        cs.functional_groups.add("carboxylate")

    # an ester consumes the partner alcohol/phenol OH -> don't claim them as FREE groups
    if "ester" in cs.functional_groups:
        cs.functional_groups.discard("phenol")
        cs.functional_groups.discard("alcohol")
        cs.notes.append("suppressed free OH claims: partner is esterified")

    # ---------- elements ----------
    for el, pat in ELEMENT_MORPHEMES.items():
        if _f(pat, low):
            cs.elements.add(el)
    if {"ester","carboxylate","carboxylic_acid","acetal","ketone","alcohol","phenol"} & cs.functional_groups:
        cs.elements.add("O")
    if {"dithioacetal","dithioketal"} & cs.functional_groups:
        cs.elements.add("S")
    if {"amide","amine","nitro"} & cs.functional_groups:
        cs.elements.add("N")
    return cs

# OCR digit repair — a VARIANT helper (not a claim); used by the resolver cell
def repair_ocr_digits(name: str) -> str:
    s = re.sub(r"(?<=[\d,\-])l(?=[\-\s]|one|ol|$)", "1", str(name))
    s = re.sub(r"(?<=\d)O(?=\d)", "0", s)
    return s

if __name__ == "__main__":
    battery = [
        "Acetaldehyde dioctyl mercaptal", "Acetic acid, o-allyl-p-cresol ester",
        "Abietic acid, ester with 2-allyl-4-hydroxy-3-methyl-2-cyclopenten-l-one",
        "Acetaldehyde, allyl 2-ethylhexyl acetal", "Abietic acid",
        "Abietic acid, ethyl ester", "Acetic acid, sodium salt",
        "Ammonium dinitro-o-cresylate", "Cinnamamide",
        "Cinnamamide, p-chloro-N,N-diethyl-", "1,2-Tetradecanediol",
        "Benzaldehyde diethyl acetal", "p-Nitrobenzoic acid, ethyl ester",
        "Camphor", "Sodium benzoate", "o-Cresol", "Succinic acid, monoethyl ester",
    ]
    for n in battery:
        c = extract_claims(n)
        print(f"{n}\n    conn={c.connectivity:7s} FG={sorted(c.functional_groups)} elem={sorted(c.elements)}")

Acetaldehyde dioctyl mercaptal
    conn=single  FG=['dithioacetal'] elem=['S']
Acetic acid, o-allyl-p-cresol ester
    conn=single  FG=['ester'] elem=['O']
Abietic acid, ester with 2-allyl-4-hydroxy-3-methyl-2-cyclopenten-l-one
    conn=single  FG=['ester'] elem=['O']
Acetaldehyde, allyl 2-ethylhexyl acetal
    conn=single  FG=['acetal'] elem=['O']
Abietic acid
    conn=single  FG=['carboxylic_acid'] elem=['O']
Abietic acid, ethyl ester
    conn=single  FG=['ester'] elem=['O']
Acetic acid, sodium salt
    conn=salt    FG=['carboxylate'] elem=['O']
Ammonium dinitro-o-cresylate
    conn=salt    FG=['nitro'] elem=['N']
Cinnamamide
    conn=single  FG=['amide'] elem=['N']
Cinnamamide, p-chloro-N,N-diethyl-
    conn=single  FG=['amide'] elem=['Cl', 'N']
1,2-Tetradecanediol
    conn=single  FG=['alcohol'] elem=['O']
Benzaldehyde diethyl acetal
    conn=single  FG=['acetal'] elem=['O']
p-Nitrobenzoic acid, ethyl ester
    conn=single  FG=['ester'] elem=['N', 'O']
Camphor
    conn=single  FG=[

In [6]:
import os
print("Working directory:", os.getcwd())
print("Files here:", [f for f in os.listdir() if f.endswith(".py")])

Working directory: /mmfs1/home/ayman.akash/Ondemand/1947 + 1954 + 1967
Files here: []


# Cell 5 — validate.py, with the ten cases wired in as runnable regression tests 
(the ALL REGRESSION TESTS PASS line is your canary; if a future edit breaks the mercaptal or ester case, it screams)

In [5]:
from __future__ import annotations
from dataclasses import dataclass, field
from rdkit import Chem
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")
from claims import extract_claims, ClaimSet

FG_SMARTS = {
    "ester":           "[CX3](=[OX1])[OX2][#6]",
    "carboxylic_acid": "[CX3](=O)[OX2H1]",
    "carboxylate":     "[$([CX3](=O)[O-]),$([CX3](=O)[OX2H1])]",
    "dithioacetal":    "[CX4]([SX2][#6])[SX2][#6]",
    "dithioketal":     "[CX4]([SX2][#6])[SX2][#6]",
    "acetal":          "[CX4]([OX2][#6])[OX2][#6]",
    "ketal":           "[CX4]([OX2][#6])[OX2][#6]",
    "amide":           "[NX3][CX3]=[OX1]",
    "amine":           "[NX3;!$([NX3][CX3]=[OX1]);!$([NX3](=O))]",
    "nitro":           "[$([NX3](=O)=O),$([NX3+](=O)[O-])]",
    "alcohol":         "[OX2H]",
    "phenol":          "[OX2H][c]",
    "ketone":          "[#6][CX3](=[OX1])[#6]",
    "ether":           "[OX2]([#6])[#6]",
}
_C = {k: Chem.MolFromSmarts(v) for k, v in FG_SMARTS.items()}

@dataclass
class Verdict:
    status: str                       # VERIFIED | UNVERIFIED | REJECT | PARSE_ERROR
    reasons: list = field(default_factory=list)
    checks: dict = field(default_factory=dict)

def validate(name: str, smiles: str, claims: ClaimSet | None = None) -> Verdict:
    cs = claims or extract_claims(name)
    v = Verdict(status="UNVERIFIED")

    if not smiles or str(smiles).strip().lower() in ("nan", "none", ""):
        return Verdict("PARSE_ERROR", ["empty SMILES"])
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return Verdict("PARSE_ERROR", ["RDKit cannot parse SMILES"])

    n_frag = len(Chem.GetMolFrags(mol))

    # 1) connectivity (hard)
    if cs.connectivity == "single" and n_frag > 1:
        return Verdict("REJECT", [f"name implies one molecule but SMILES has {n_frag} fragments"],
                       {"connectivity": False})
    v.checks["connectivity"] = True

    # 2) elements present (hard)
    present = {a.GetSymbol() for a in mol.GetAtoms()}
    for el in cs.elements:
        if el not in present:
            return Verdict("REJECT", [f"name implies element {el} but SMILES has none"],
                           {"element_missing": el})
    if cs.elements:
        v.checks["elements"] = True

    # 3) functional-group SMARTS (hard on mismatch; strong corroboration on match)
    fg_confirmed = []
    for fg in cs.functional_groups:
        patt = _C.get(fg)
        if patt is None:
            continue
        if mol.HasSubstructMatch(patt):
            fg_confirmed.append(fg)
        else:
            if cs.connectivity == "salt" and fg in ("carboxylate",):
                continue   # a salt returned in neutral/parent form is acceptable
            return Verdict("REJECT", [f"name implies {fg} but SMILES lacks that group"],
                           {"fg_missing": fg})
    v.checks["functional_groups"] = fg_confirmed

    # 4) status
    if fg_confirmed:
        v.status = "VERIFIED"
        v.reasons.append(f"structure confirms: {', '.join(sorted(fg_confirmed))}")
    else:
        v.status = "UNVERIFIED"
        v.reasons.append("passed all applicable checks; no strong functional-group cue to confirm")
    return v


if __name__ == "__main__":
    CASES = [
        ("Acetaldehyde dioctyl mercaptal", "CCCCCCCSC(C)SCCCCCCCC", "VERIFIED"),
        ("Acetaldehyde dioctyl mercaptal", "CCCCCCCOCSCCCCCCCC",    "REJECT"),
        ("Acetic acid, o-allyl-p-cresol ester", "CC(=O)Oc1ccc(C)cc1CC=C", "VERIFIED"),
        ("Acetic acid, o-allyl-p-cresol ester", "CC(O)=O.Cc1ccc(O)c(CC=C)c1", "REJECT"),
        ("Sodium benzoate", "O=C([O-])c1ccccc1.[Na+]", "VERIFIED"),
        ("Acetic acid, sodium salt", "CC(=O)[O-].[Na+]", "VERIFIED"),
        ("Cinnamamide, p-chloro-N,N-diethyl-", "CCN(CC)C(=O)/C=C/c1ccc(Cl)cc1", "VERIFIED"),
        ("Camphor", "CC1(C)C2CCC1(C)C(=O)C2", "UNVERIFIED"),
        ("p-Nitrobenzoic acid, ethyl ester", "CCOC(=O)c1ccc([N+](=O)[O-])cc1", "VERIFIED"),
        ("Benzaldehyde diethyl acetal", "CCOC(OCC)c1ccccc1", "VERIFIED"),
    ]
    ok = True
    for name, smi, exp in CASES:
        got = validate(name, smi)
        flag = "PASS" if got.status == exp else "**FAIL**"
        if got.status != exp: ok = False
        print(f"[{flag}] want={exp:11s} got={got.status:11s} | {name[:42]:42s} | {got.reasons[0][:60]}")
    print("\nALL REGRESSION TESTS PASS" if ok else "\nSOME TESTS FAILED")

ModuleNotFoundError: No module named 'claims'